# Lab 001 · Colab：观察第一个 Token

今天只完成三件事：确认 GPU、运行一次小模型、保存实际计时结果。

1. 在 Colab 的“运行时 → 更改运行时类型”选择可用的 **GPU**，然后连接。
2. 按顺序运行单元格，先到“第一次真实生成”为止。
3. 把 GPU 型号、输入/输出 Token 数、首 Token 延迟和后续平均间隔发回聊天。

本实验直接调用 Transformers，不需要模型 API 或密钥。计时从 GPU 输入就绪、调用 `generate` 前开始，到 Token ID 在 CPU 回调可用为止；不含下载、加载、分词、网络和服务端排队，包含生成框架、采样与拷贝开销。它不是纯 Prefill Kernel 时间，也不是 HTTP 端到端 TTFT。

Colab 的 GPU 类型与可用性不保证固定。若没有 GPU，停在检查单元格，把提示发回来。笔记本只在实际运行后产生数据。


## 1. 检查 GPU

这是我们今天的第一条环境证据。`torch.cuda.is_available()` 必须是 True。模型下载或依赖安装都放在检查之后。


In [ ]:
import sys
import platform
import subprocess
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
assert sys.version_info >= (3, 10), "本笔记本需要 Python 3.10 或更新版本"
assert torch.cuda.is_available(), "请在运行时设置中选择 GPU 后重新连接；若没有额度，先停止并回到聊天"
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("GPU memory GiB:", round(props.total_memory / 1024**3, 2))
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout
print(gpu_info)


## 2. 安装实验依赖

固定 Transformers 版本，保留 Colab 提供的 PyTorch。首次运行会联网下载依赖和模型。

若 Colab 提示重启会话，重启后重新执行 GPU 检查，再从第 3 步继续。若之前已经导入其他版本的 Transformers，也先重启会话。


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.1"])


## 3. 加载小模型并记录版本

使用 Qwen2.5-0.5B-Instruct，FP16、单 GPU、SDPA Attention。模型大小适合从小实验起步；不要求先部署 vLLM。

首次从 main 解析模型 commit，并在同一次实验固定 Tokenizer 和权重版本。以后复现时，将保存的 commit 填到 `MODEL_REVISION`。下载和加载耗时不计入推理延迟。


In [ ]:
import json
import time
import uuid
import hashlib
import importlib.metadata as metadata
from pathlib import Path
from datetime import datetime, timezone
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

assert transformers.__version__ == "5.20.1", "请重启会话，加载刚安装的固定版本"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = None  # 复现时可以填入已保存的 commit SHA
MODEL_REVISION = MODEL_REVISION or model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
    attn_implementation="sdpa", trust_remote_code=False, use_safetensors=True,
).to("cuda").eval()

run_dir = Path("/content/ai-infra-lab/lab001") / uuid.uuid4().hex
run_dir.mkdir(parents=True, exist_ok=False)
environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda, "gpu": props.name,
    "gpu_memory_bytes": props.total_memory, "nvidia_smi": gpu_info,
    "model": MODEL_ID, "model_revision": MODEL_REVISION,
    "dtype": str(model.dtype), "attention": "sdpa", "batch_size": 1,
    "dependencies": {p: metadata.version(p) for p in ("transformers", "huggingface-hub", "tokenizers", "safetensors")},
    "chat_template_sha256": hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest(),
    "timing_scope": "host wall-clock generate invocation to CPU token callback; not HTTP TTFT or pure GPU time",
    "cache_scope": "KV cache reused within each request; no supplied cross-request prefix cache",
}
(run_dir / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2))
print("Model revision:", MODEL_REVISION)
print("Model loaded on:", model.device)
print("Results directory:", run_dir)


## 4. 准备计时器

框架首先把输入交给 streamer，随后逐个交出新 Token。我们跳过输入，只记录生成 Token ID 的到达时间，避免把一个文字片段误当作一个 Token。

只支持当前的单请求、单序列、贪心生成。每次记录实际 Token 数，并检查回调 Token 与最终输出是否一致。回调本身会产生额外开销，后续比较必须保持相同方法。


In [ ]:
class TokenTimer:
    def __init__(self):
        self.prompt_seen = False
        self.events = []
        self.started = None
        self.ended = False

    def put(self, value):
        ids = value.reshape(-1).tolist()
        if not self.prompt_seen:
            self.prompt_seen = True
            return
        if len(ids) != 1:
            raise RuntimeError("当前计时器只支持每次一个新 Token")
        self.events.append({"token_id": ids[0], "received_s": time.perf_counter() - self.started})

    def end(self):
        self.ended = True


def measure(prompt, label, max_new_tokens=64, fixed_length=False, warmup=False):
    messages = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to("cuda")
    input_count = int(inputs["input_ids"].shape[-1])
    assert input_count + max_new_tokens <= min(model.config.max_position_embeddings, 4096), "首轮实验限制总长度到 4096"
    timer = TokenTimer()
    record = {
        "id": uuid.uuid4().hex, "at_utc": datetime.now(timezone.utc).isoformat(),
        "label": label, "prompt": prompt, "rendered_prompt": rendered,
        "input_tokens": input_count, "max_new_tokens": max_new_tokens,
        "min_new_tokens": max_new_tokens if fixed_length else 0,
        "do_sample": False, "use_cache": True, "warmup": warmup,
        "success": False, "events": [], "error": None,
    }
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    timer.started = time.perf_counter()
    try:
        with torch.inference_mode():
            output = model.generate(
                **inputs, do_sample=False, num_beams=1, num_return_sequences=1,
                max_new_tokens=max_new_tokens,
                min_new_tokens=max_new_tokens if fixed_length else 0,
                use_cache=True, streamer=timer, pad_token_id=tokenizer.eos_token_id,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - timer.started
        generated = output[0, input_count:].tolist()
        assert timer.ended and generated == [e["token_id"] for e in timer.events], "回调与输出不一致，不能使用这些计时"
        assert generated, "没有生成 Token"
        times = [e["received_s"] for e in timer.events]
        record.update({
            "success": True, "output_tokens": len(generated),
            "generated_token_ids": generated,
            "text": tokenizer.decode(generated, skip_special_tokens=True),
            "first_token_s": times[0], "generate_elapsed_s": elapsed,
            "mean_later_token_s": (times[-1] - times[0]) / (len(times) - 1) if len(times) > 1 else None,
            "stop_observation": "length_limit" if len(generated) >= max_new_tokens else "early_stop",
        })
    except Exception as exc:
        record["error"] = {"type": type(exc).__name__, "message": str(exc)}
        raise
    finally:
        record["events"] = timer.events
        record["peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
        (run_dir / (record["id"] + ".json")).write_text(json.dumps(record, ensure_ascii=False, indent=2))
    return record


def show(record):
    print("输入 / 输出 Token:", record["input_tokens"], "/", record["output_tokens"])
    print("本地首 Token 延迟: %.2f ms" % (record["first_token_s"] * 1000))
    value = record["mean_later_token_s"]
    print("后续 Token 平均间隔:", "不足两个 Token" if value is None else "%.2f ms" % (value * 1000))
    print("generate 完整耗时: %.3f s" % record["generate_elapsed_s"])
    print("停止情况:", record["stop_observation"])
    print("正文:", record["text"])


## 5. 预热，再完成第一次真实生成

预热结果单独保存，不混入正式测量。第一次只检查是否成功；不要从一个样本判断 GPU 性能。

先预测：如果输入变长，而输出长度相同，首 Token 延迟与后续 Token 平均间隔，哪一项更可能变化？目前不需要答对，先写下直觉。


In [ ]:
_ = measure("你好，请介绍一下自己。", "warmup", max_new_tokens=8, warmup=True)
first = measure("请用中文列出五条 GPU 的用途。", "first-real-request", max_new_tokens=64)
show(first)
print("GPU:", environment["gpu"])
print("结果目录:", run_dir)


**第一次先停在这里。** 将上面输出发回聊天，我们先解释实际结果，再决定是否继续下面的对照。

首 Token 可能是特殊 Token，不一定已经变成可见文字。后续平均间隔来自 Token ID 的观测时间，包含框架与回调开销，不是纯 GPU Decode 时间。这里没有 HTTP 排队或 vLLM Continuous Batching，结果不代表生产服务性能。


## 6. 可选：三档输入长度对照

确认首轮成功后，将 `RUN_COMPARISON` 改为 True 再运行。使用三组不同长度的真实文本，报告包含 Chat Template 后的实际输入 Token 数。每组先预热，再以固定随机种子交错执行五次；每次固定生成 32 Token，抑制提前 EOS，因此这是性能工作负载，不用它评价自然回答质量。

不传入跨请求 KV，每次请求内部仍使用 KV。硬件预热、分配器与功耗状态仍可能影响结果。


In [ ]:
RUN_COMPARISON = False
comparison = []
if RUN_COMPARISON:
    import random
    task = "请列出十条 GPU 性能优化建议，每条用一句话说明。"
    background = "GPU 可并行处理大量数据，计算、显存容量与带宽都会影响性能。\n"
    prompts = {
        "short": task,
        "medium": "以下背景仅供参考：\n" + background * 16 + task,
        "long": "以下背景仅供参考：\n" + background * 64 + task,
    }
    for label, prompt in prompts.items():
        measure(prompt, label + "-warmup", max_new_tokens=32, fixed_length=True, warmup=True)
    order = list(prompts) * 5
    random.Random(42).shuffle(order)
    for index, label in enumerate(order):
        record = measure(prompts[label], label, max_new_tokens=32, fixed_length=True)
        comparison.append(record)
        print(index + 1, label, record["input_tokens"], "tokens, first %.2f ms" % (record["first_token_s"] * 1000))
else:
    print("对照尚未运行：先把第一次真实生成结果发回聊天。")


In [ ]:
if comparison:
    import csv
    import statistics
    import matplotlib.pyplot as plt
    rows = []
    for label in ("short", "medium", "long"):
        group = [r for r in comparison if r["label"] == label]
        row = {
            "label": label, "n": len(group), "input_tokens": group[0]["input_tokens"],
            "output_tokens": group[0]["output_tokens"],
            "median_first_ms": statistics.median(r["first_token_s"] for r in group) * 1000,
            "median_later_ms": statistics.median(r["mean_later_token_s"] for r in group) * 1000,
        }
        rows.append(row)
        print(row)
    with (run_dir / "summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, key, title in zip(axes, ("first_token_s", "mean_later_token_s"), ("First token (host wall time)", "Mean later-token interval")):
        ax.scatter([r["input_tokens"] for r in comparison], [r[key] * 1000 for r in comparison], alpha=0.6)
        ax.set(xlabel="Actual input tokens", ylabel="ms", title=title)
    fig.tight_layout()
    fig.savefig(run_dir / "latency.png", dpi=150)
    plt.show()


## 7. 下载结果

运行下面的单元格下载一个结果压缩包。Colab 会话的文件可能随运行时删除而丢失，实验后保存到本地；提交项目时检查其中提示词和输出是否适合公开。


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
files.download(archive)


## 学习记录

- 我实际分配到的 GPU：待填。
- 输入长度变化后，我观察到了什么：待填。
- 这个实验能支持的结论：待填。
- 仍无法解释的现象：待填。
- 下一步：再研究 KV Cache、计算量与计时边界；需要时转到服务接口实验。

## 参考资料

访问日期：2026-10-17。
- [Google Colab FAQ：GPU、上传笔记本与保存结果](https://research.google.com/colaboratory/faq.html)
- [Qwen2.5-0.5B-Instruct 官方模型卡](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- [Transformers 5.20.1](https://pypi.org/project/transformers/5.20.1/)
- [固定版本生成源码：streamer 接收输入与生成 Token](https://github.com/huggingface/transformers/blob/v5.20.1/src/transformers/generation/utils.py)

验证状态：本地检查笔记本结构、Python 语法和计时/统计逻辑；尚未在用户的 Colab GPU 上执行，不包含预填性能结果。
